# Amharic Automatic Speech Recognition (ASR)
### Fine-tuning Whisper for Amharic Transcription

This notebook provides a complete, step-by-step workflow for transcribing Amharic audio and fine-tuning the OpenAI Whisper model to improve its accuracy. 

**Workflow:**
1. **Import Existing Model:** Load the pre-trained Whisper model.
2. **Initial Inference:** Transcribe a sample audio using the base model.
3. **Dataset Preparation:** Load and process an Amharic dataset for training.
4. **Fine-tuning:** Train the model to better understand Amharic speech.
5. **Final Inference:** Verify the improvements with the fine-tuned model.

### 🛠️ How to use this notebook
- **Hardware Accelerator:** For training, go to `Runtime` -> `Change runtime type` and select **GPU** (T4 or better).
- **Uploading Files:** Click the folder icon on the left sidebar to upload your audio files and CSV datasets.
- **Form Fields:** Use the forms on the right side of the code cells to adjust parameters without editing the code directly.

## 0. Setup and Installation
First, we install the necessary libraries and set up our environment. We need `transformers` for the model, `datasets` for handling audio data, and `evaluate` for calculating accuracy metrics like WER (Word Error Rate).

In [ ]:
# @title Install Dependencies
# @markdown This may take a minute or two.
!pip install torch transformers[torch] datasets accelerate evaluate jiwer librosa soundfile PyYAML -q

In [ ]:
# @title Standard Imports
import os
import torch
import evaluate
import numpy as np
from dataclasses import dataclass
from typing import Any, List, Dict, Union
from datasets import Audio, DatasetDict, load_dataset
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    pipeline,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

### (Optional) Connect to Google Drive
Highly recommended if you want to save your model checkpoints and large datasets permanently. If you don't use Drive, your files will be deleted when the session ends.

In [ ]:
# @title Google Drive Connection
from google.colab import drive
mount_drive = False # @param {type:"boolean"}
if mount_drive:
    drive.mount('/content/drive')
    print("Google Drive mounted.")

## 1. Import Existing Model
We load the `whisper-small` model from OpenAI. This model is already quite capable but can be significantly improved with Amharic-specific data. The `Processor` handles both the audio preprocessing (Feature Extractor) and the text decoding (Tokenizer).

In [ ]:
# @title Load Model and Processor
model_id = "openai/whisper-small" # @param ["openai/whisper-tiny", "openai/whisper-base", "openai/whisper-small", "openai/whisper-medium"]

print(f"Loading {model_id}...")
processor = WhisperProcessor.from_pretrained(model_id, language="am", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_id).to(device)

# Set generation parameters to Amharic
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

print(f"Model {model_id} is ready for use.")

## 2. Generate Transcript from Sample Audio
Let's test the base model before any fine-tuning. This gives us a baseline for comparison.

**Note:** If you haven't uploaded an audio file, this cell will create a dummy noise file to ensure the code runs.

In [ ]:
# @title Transcription with Base Model

def transcribe(audio_path, model_to_use, processor_to_use):
    # Create ASR pipeline for easy inference
    pipe = pipeline(
        "automatic-speech-recognition",
        model=model_to_use,
        tokenizer=processor_to_use.tokenizer,
        feature_extractor=processor_to_use.feature_extractor,
        device=0 if torch.cuda.is_available() else -1,
    )
    
    # Specifically configure for Amharic transcription
    generate_kwargs = {"language": "am", "task": "transcribe"}
    
    result = pipe(audio_path, generate_kwargs=generate_kwargs)
    return result["text"]

sample_audio_path = "sample_audio.wav" # @param {type:"string"}

# Check for existence or create dummy
if not os.path.exists(sample_audio_path):
    print(f"Audio file {sample_audio_path} not found. Creating a dummy file for demonstration...")
    import soundfile as sf
    sr = 16000
    dummy_data = np.random.uniform(-1, 1, sr * 2) # 2 seconds of noise
    sf.write(sample_audio_path, dummy_data, sr)

print("Transcribing...")
initial_transcript = transcribe(sample_audio_path, model, processor)
print("="*30)
print(f"Initial Transcript: {initial_transcript}")
print("="*30)

## 3. Import Dataset to Refine the Model
To fine-tune the model, we need a dataset of audio recordings and their correct Amharic transcripts. 

### 📄 CSV Format
Your CSV should look like this:
| audio_path | transcript |
|------------|------------|
| data/audio1.wav | ሰላም ለእናንተ ይሁን |
| data/audio2.wav | እንኳን ደህና መጣችሁ |

In [ ]:
# @title Load and Prepare Dataset

train_csv = "train.csv" # @param {type:"string"}
val_csv = "validation.csv" # @param {type:"string"}
audio_col = "audio_path" # @param {type:"string"}
text_col = "transcript" # @param {type:"string"}

def prepare_dataset_fn(batch):
    # Resample audio to 16kHz as required by Whisper
    audio = batch[audio_col]
    
    # Compute log-Mel input features
    batch["input_features"] = processor.feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    
    # Encode target text to label IDs
    batch["labels"] = processor.tokenizer(batch[text_col]).input_ids
    return batch

# Create dummy CSV files if they don't exist, for demonstration purposes
if not os.path.exists(train_csv):
    import pandas as pd
    pd.DataFrame({
        audio_col: [sample_audio_path],
        text_col: ["ሰላም ልዑል"] 
    }).to_csv(train_csv, index=False)
    print(f"Created dummy {train_csv}")

if not os.path.exists(val_csv):
    import pandas as pd
    pd.DataFrame({
        audio_col: [sample_audio_path],
        text_col: ["ሰላም ልዑል"]
    }).to_csv(val_csv, index=False)
    print(f"Created dummy {val_csv}")

if os.path.exists(train_csv) and os.path.exists(val_csv):
    print("Loading dataset...")
    dataset = load_dataset("csv", data_files={"train": train_csv, "test": val_csv})
    
    # Important: Whisper expects 16,000Hz mono audio
    dataset = dataset.cast_column(audio_col, Audio(sampling_rate=16000))
    
    print("Preprocessing dataset (converting audio to features)... ")
    dataset = dataset.map(
        prepare_dataset_fn, 
        remove_columns=dataset["train"].column_names, 
        num_proc=1
    )
    dataset_ready = True
    print("Dataset is ready.")
else:
    dataset_ready = False
    print(f"Dataset files not found. Please upload '{train_csv}' and '{val_csv}' or update the paths above.")

## 4. Train to Refine the Model
We use the `Seq2SeqTrainer` to fine-tune Whisper. This process will adjust the model's weights to better match Amharic phonetic patterns.

**Note:** Training is a computationally intensive task. Make sure you are using a GPU runtime.

In [ ]:
# @title Fine-tuning the Model

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Data collator that will dynamically pad the inputs received."""
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels since they have different lengths and need different padding methods
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")
        
        # Replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        
        # If bos token is appended in previous steps, remove it as it's added by the model
        if labels.shape[1] > 0 and (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

if dataset_ready:
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
    wer_metric = evaluate.load("wer")

    def compute_metrics(pred):
        pred_ids = pred.predictions
        label_ids = pred.label_ids
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
        pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
        label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)
        # Calculate Word Error Rate (lower is better)
        wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
        return {"wer": wer}

    # Training Configuration
    training_args = Seq2SeqTrainingArguments(
        output_dir="./whisper-small-amharic", 
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2, # effectively batch size 16
        learning_rate=1e-5,
        warmup_steps=50,
        max_steps=100, # Set low for demonstration; increase for better results
        gradient_checkpointing=True,
        fp16=torch.cuda.is_available(),
        evaluation_strategy="steps",
        per_device_eval_batch_size=8,
        predict_with_generate=True,
        generation_max_length=225,
        save_steps=50,
        eval_steps=50,
        logging_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="wer",
        greater_is_better=False,
        report_to=[],
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
    )

    print("Starting training...")
    trainer.train()
    print("Training completed successfully!")
else:
    print("Training skipped because the dataset was not found.")

## 5. Infer the Trained Model
Now let's see the improvement. We use the same sample audio to compare the output of the fine-tuned model against the original baseline.

In [ ]:
# @title Transcription with Fine-tuned Model
print("Transcribing with the new fine-tuned model...")

if 'trainer' in locals():
    # Use the best model weights found during the training process
    final_transcript = transcribe(sample_audio_path, trainer.model, processor)
else:
    print("Fine-tuned model not found in memory. Using the original model instead.")
    final_transcript = transcribe(sample_audio_path, model, processor)

print("="*30)
print(f"Initial Transcript (Base Model): {initial_transcript}")
print(f"Final Transcript (Fine-tuned Model): {final_transcript}")
print("="*30)